# Nemotron post-training with **NeMo-RL** — GRPO on the 0.85 checkpoint (OFF-Kaggle)

NeMo-RL (`github.com/NVIDIA-NeMo/RL`) is NVIDIA's scalable RL post-training toolkit with native
Nemotron support. It is **cluster-scale** (Ray + multi-GPU + vLLM serving) and **cannot run inside
a Kaggle offline single-GPU notebook**. Run this on your **own GPU box / cloud** (1×H100 minimum,
better 2–8×), then **upload only the resulting rank-≤32 LoRA adapter** as your Kaggle submission —
the contest only checks the adapter, not how it was trained.

## Why GRPO here
SFT imitates the teacher (ceiling ≈ teacher accuracy). GRPO optimizes the **actual metric** via a
verifier reward, so it can exceed the teacher — the path past the 0.85 SFT plateau. *LoRA Without
Regret*: RL matches full-FT even at rank 1, so the rank-32 cap is a non-issue for RL.

## Flow (this notebook writes the files; you run the commands)
1. Install NeMo-RL.
2. `prepare_data.py` → `train.jsonl` (synthetic-perfect + train.csv, with ground-truth answers).
3. `wonderland_env.py` → custom Environment whose reward = the **official metric** (`compare_answer`),
   plus a small format bonus for `<think>…</think>\boxed{}`.
4. `grpo_nemotron_lora.yaml` → GRPO + LoRA(r=32) config, vLLM generation, warm-started from the 0.85 adapter.
5. Launch `run_grpo.py`; extract the LoRA adapter → `submission.zip`.

> Config keys follow the NeMo-RL GRPO guide (docs.nvidia.com/nemo/rl). Pin the repo commit you
> install and reconcile key names with that commit — the API moves. Treat this as a working template.


In [ ]:
# ── 1. Install NeMo-RL (run on your OFF-Kaggle GPU box) ──
# Uncomment to run. Needs CUDA + uv. Pin a commit so the config keys below stay valid.
INSTALL = r'''
git clone https://github.com/NVIDIA-NeMo/RL.git nemo-rl
cd nemo-rl
git checkout <PIN_A_RELEASE_TAG>          # e.g. a tagged release; record it
uv venv && source .venv/bin/activate
uv sync                                   # installs nemo-rl + vllm + megatron/dtensor backends
# HF auth for the base model (gated):
huggingface-cli login
'''
print(INSTALL)
print("Run the above in a shell on the GPU box, then continue in that repo's env.")


In [ ]:
# ── 2. Build train.jsonl for NeMo-RL (prompt + ground-truth answer per row) ──
# Pool = synthetic-perfect (from gpt5_trace_gen/synthetic_hard.csv) + the competition train.csv.
# NeMo-RL GRPO needs only (input, ground_truth); the reward env grades rollouts against it.
import json, csv, os, re

OUT = "train.jsonl"
PROMPT_SUFFIX = ("\nPlease put your final answer inside `\\boxed{}`. "
                 "For example: `\\boxed{your answer}`")

rows = []

# (a) competition train.csv — perfect answers, exact distribution
TRAIN_CSV = os.environ.get("TRAIN_CSV", "train.csv")   # the Kaggle train.csv
if os.path.exists(TRAIN_CSV):
    for r in csv.DictReader(open(TRAIN_CSV, encoding="utf-8")):
        rows.append({"input": r["prompt"] + PROMPT_SUFFIX, "ground_truth": str(r["answer"]),
                     "task_name": "wonderland"})

# (b) synthetic-perfect hard categories (solver-verified)
SYN_CSV = os.environ.get("SYN_CSV", "synthetic_hard.csv")
if os.path.exists(SYN_CSV):
    for r in csv.DictReader(open(SYN_CSV, encoding="utf-8")):
        rows.append({"input": r["prompt"] + PROMPT_SUFFIX, "ground_truth": str(r["answer"]),
                     "task_name": "wonderland"})

with open(OUT, "w", encoding="utf-8") as f:
    for r in rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print(f"wrote {len(rows)} prompts -> {OUT}")
print("NeMo-RL data keys: input_key='input', output_key not needed (RL), ground_truth via env.")


In [ ]:
# ── 3. Custom reward Environment (writes nemo-rl/nemo_rl/environments/wonderland_env.py) ──
# Reward = R_accuracy (official metric, +1 if boxed answer matches ground truth)
#        + 0.2 * R_format (<think>...</think> before \boxed{}). Mirrors NeMo-RL's recipe.
WONDERLAND_ENV = r'''
import re, math
try:
    from nemo_rl.environments.interfaces import Environment
except Exception:                       # allow import outside the repo for linting
    class Environment: ...

_BX = re.compile(r"\\boxed\{([^{}]*)\}")

def extract_boxed(t):
    i = t.rfind("\\boxed{")
    if i == -1:
        m = _BX.findall(t); return m[-1].strip() if m else None
    d, j = 1, i + 7
    while j < len(t) and d > 0:
        if t[j] == "{": d += 1
        elif t[j] == "}": d -= 1
        j += 1
    return t[i + 7:j - 1].strip()

def compare_answer(stored, predicted):
    """Official metric (syn_datagen/reasoning.py)."""
    if predicted is None: return False
    stored, predicted = str(stored).strip(), str(predicted).strip()
    if re.fullmatch(r"[01]+", stored):
        return predicted.lower() == stored.lower()
    try:
        return math.isclose(float(stored), float(predicted), rel_tol=1e-2, abs_tol=1e-5)
    except Exception:
        return predicted.lower() == stored.lower()

def _format_ok(t):
    return ("</think>" in t and "\\boxed{" in t and t.find("</think>") < t.rfind("\\boxed{"))

class WonderlandEnvironment(Environment):
    """Grades a generated rollout against the ground-truth answer."""
    def step(self, state, action):
        gt = state.get("ground_truth", "")
        pred = extract_boxed(action)
        r_acc = 1.0 if compare_answer(gt, pred) else 0.0
        r_fmt = 1.0 if _format_ok(action) else 0.0
        reward = r_acc + 0.2 * r_fmt
        return state, reward
'''
import os
path = os.path.join("nemo-rl", "nemo_rl", "environments", "wonderland_env.py")
os.makedirs(os.path.dirname(path), exist_ok=True)
open(path, "w", encoding="utf-8").write(WONDERLAND_ENV)
print("wrote", path)
print("Register it: environments.wonderland.type=WonderlandEnvironment in the config (next cell).")


In [ ]:
# ── 4. GRPO + LoRA(r=32) config (writes grpo_nemotron_lora.yaml) ──
# Warm-start from the 0.85 LoRA: merge it into the base first (scripts/merge below) OR pass it as
# the initial policy weights if your NeMo-RL commit supports adapter init. Keys follow the GRPO guide.
CONFIG = r'''
policy:
  model_name: "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"   # or your 0.85-merged checkpoint dir
  precision: "bfloat16"
  max_total_sequence_length: 8192
  dtensor_cfg:
    enabled: true
    lora_cfg:
      enabled: true
      r: 32                       # SUBMISSION CAP — do not exceed
      lora_alpha: 64
      lora_dropout: 0.0
      # Nemotron-H linears (verify exact names via model.named_modules on your commit):
      target_modules: ["q_proj","k_proj","v_proj","o_proj","in_proj","out_proj","up_proj","down_proj"]

generation:
  backend: "vllm"
  temperature: 1.0
  top_p: 1.0
  top_k: 0
  max_new_tokens: 3000

grpo:
  num_prompts_per_step: 32
  num_generations: 16           # rollouts per prompt (group-relative baseline)
  max_total_sequence_length: 8192
  learning_rate: 1.0e-6         # Dr.GRPO-style low LR
  kl_beta: 0.0                  # no reference model (memory + post-DeepSeek-R1 standard)
  loss_type: "dr_grpo"          # if available on your commit; else grpo + scale_rewards=false
  scale_rewards: false
  mask_truncated_completions: true
  num_train_steps: 400
  optimizer: "adamw"
  grad_clip: 1.0

data:
  train:
    data_path: "train.jsonl"
    input_key: "input"
    env_name: "wonderland"

environments:
  wonderland:
    type: "WonderlandEnvironment"

cluster:
  num_nodes: 1
  gpus_per_node: 1              # raise for real throughput (2-8x H100)
'''
open("grpo_nemotron_lora.yaml", "w", encoding="utf-8").write(CONFIG)
print("wrote grpo_nemotron_lora.yaml")
print("NOTE: reconcile keys with `examples/configs/recipes/llm/grpo-*.yaml` from YOUR pinned commit.")


## 5. Launch + package the adapter for Kaggle

```bash
# (optional) warm-start: merge your 0.85 LoRA into the base so GRPO continues from it
python tools/merge_lora.py \
    --base nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16 \
    --adapter /path/to/0.85-adapter --out ./nemotron-0p85-merged
# then set policy.model_name: ./nemotron-0p85-merged in the yaml

# launch GRPO (single node; scale gpus_per_node for speed)
cd nemo-rl
uv run examples/run_grpo.py \
    --config ../grpo_nemotron_lora.yaml \
    policy.dtensor_cfg.lora_cfg.enabled=true \
    data.train.data_path=../train.jsonl \
    grpo.num_train_steps=400

# watch: reward should rise; reward_accuracy is the metric proxy. Log per-category if possible.
```

### Extract the LoRA adapter → submission.zip
NeMo-RL saves a checkpoint; export just the **adapter** (rank-32) and zip the two files at root:
```bash
python tools/export_lora.py --ckpt outputs/<run>/checkpoints/step_400 --out ./adapter_out
cd adapter_out && zip ../submission.zip adapter_config.json adapter_model.safetensors
```
Then upload `submission.zip` to Kaggle (LoRA-only, rank ≤ 32 — same contract as the SFT path).

### Reality check
- **Compute:** 30B MoE + 16 rollouts × 8192 tokens is heavy — 1×H100 is the floor; expect hours/100s of steps.
- **Eval-gate** the GRPO adapter against the 0.85 baseline before shipping. Keep the winner.
- This is the **same loop** as the on-Kaggle **v25 RAFT** (warm-start 0.85, verifier reward, rollout→reward→update).
  Use NeMo-RL only if you have idle off-Kaggle GPUs; otherwise v25 is the practical path.
